# Libraries

In [1]:
import pandas as pd
import requests 
from bs4 import BeautifulSoup

# Obtendo Conexão

In [ ]:
req = requests.get("https://sol.sbc.org.br/busca/index.php/integrada/results?query=&archiveIds%5B%5D=1&archiveIds%5B%5D=2&archiveIds%5B%5D=3&isAdvanced=1&field-10%5B%5D=por&field-10%5B%5D=eng&searchPage=1#records")
if  req.status_code == 200:
    print("Conexão bem sucedida!")

In [ ]:
soup = BeautifulSoup(req.content, 'html.parser')
#print(soup.prettify())

Obtendo a última página

In [ ]:
pagination_div = soup.find("div", class_="pagination_results")

for a in pagination_div.find_all("a", href=True):
    link = a["href"]

print(link)    

In [ ]:
start_delimiter = "searchPage="
end_delimiter = "#records"


start_index = link.find(start_delimiter)
if start_index != -1:
    start_index += len(start_delimiter)
    
    end_index = link.find(end_delimiter, start_index)
    
    if end_index != -1:
        last_page = link[start_index:end_index]
        print(f"Última Página: {last_page}")
    else:
        print("Delimitador final não encontrado.")
else:
    print("Delimitador inicial não encontrado.")



Obtendo Informações dos artigos

In [ ]:
list_page = []

df = pd.DataFrame(columns=['Category', 'Title', 'URL_Title', 'Authors', 'Event', 'Date'])

for page in range(1, int(last_page) + 1):
    
    url = f"https://sol.sbc.org.br/busca/index.php/integrada/results?query=&archiveIds%5B%5D=1&archiveIds%5B%5D=2&archiveIds%5B%5D=3&isAdvanced=1&field-10%5B%5D=por&field-10%5B%5D=eng&searchPage={page}"
    print("Página", page, " - ", url)
    
    req = requests.get(url)
    if  req.status_code == 200:
        print("Conexão bem sucedida!")
    
    soup = BeautifulSoup(req.content, 'html.parser')
    #print(soup.prettify())

    list_category = soup.find_all(class_="archive_title")
    list_title = soup.find_all(class_="title")
    list_url_title = soup.find_all(class_="record_title")
    list_authors = soup.find_all(class_="author")
    list_event = soup.find_all(class_="archive_serie")
    list_date = soup.find_all(class_="list_record_date")
 
    if len(list_title) != 26:
        list_page.append(page)
        print("Página com menos de 25 registros:", page)


    for category, title, url_title, authors, event, date in zip(list_category, list_title, list_url_title, list_authors, list_event, list_date):
        category = category.get_text()
        title = title.get_text()
        url_title = url_title.get('href')
        authors = authors.get_text()
        event = event.get_text()    
        date = date.get_text(strip=True)
    
        df = df._append({'Category': category, 'Title': title, 'URL_Title': url_title, 'Authors': authors, 'Event': event, 'Date': date}, ignore_index=True)

In [ ]:
df.head()

In [ ]:
# Mantendo apenas Anais de Evento e Periódicos
df = df[df['Category'].str.contains("ANAIS DE EVENTO|PERIÓDICOS", regex=True, na=False)]

In [ ]:
df['Box'] = None

df.loc[0:3999, 'Box'] = "Caixa 1"
df.loc[4000:7999, 'Box'] = "Caixa 2"
df.loc[8000:11999, 'Box'] = "Caixa 3"
df.loc[12000:15999, 'Box'] = "Caixa 4"
df.loc[16000:19999, 'Box'] = "Caixa 5"
df.loc[20000:23999, 'Box'] = "Caixa 6"
df.loc[24000:27999, 'Box'] = "Caixa 7"
df.loc[28000:31999, 'Box'] = "Caixa 8"
df.loc[32000:, 'Box'] = "Caixa 9"


In [ ]:
df.to_csv('sol_sbc_eventos_periodicos.csv', index=False)

Obtendo os PDF de cada artigo

In [15]:
df = pd.read_csv('sol_sbc_eventos_periodicos.csv')

df_box = df[df['Box'] == "Caixa 1"]

In [16]:
df_box.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3999 entries, 0 to 3998
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Category   3999 non-null   object
 1   Title      3999 non-null   object
 2   URL_Title  3999 non-null   object
 3   Authors    3999 non-null   object
 4   Event      3999 non-null   object
 5   Date       3999 non-null   object
 6   Box        3999 non-null   object
dtypes: object(7)
memory usage: 249.9+ KB


In [17]:
df_papers = pd.DataFrame(columns=['Event', 'Authors', 'Title', 'Abstract', 'Keywords', 'Publisher', 'URL_Paper', 'Date'])

paper_number = 1

for url in df_box['URL_Title']:
    
    print(url)
    print("Paper number:", paper_number)
    req = requests.get(url)

    soup = BeautifulSoup(req.content, 'html.parser')
    soup.prettify()

    try:
        event = soup.find(class_="cmp_breadcrumbs")
        event = event.get_text()
        #print(event)
    except AttributeError:
        event = "No event available"
    
    try:
        list_authors = soup.find(class_="item authors")
        list_authors = list_authors.get_text(strip=True)
        #print(list_authors)
    except AttributeError:
        list_authors = "No authors available"
    
    try:
        title = soup.find(class_="page_title")
        title = title.get_text(strip=True)
        #print(title)
    except AttributeError:
        title = "No title available"

    try:
        abstract = soup.find(class_="item abstract")
        abstract = abstract.get_text(strip=True)
    except AttributeError:
        abstract = "No abstract available"
        #print(abstract)

    try:
        keywords = soup.find(class_="item keywords")
        keywords = keywords.get_text(strip=True)
    except AttributeError:
        keywords = "No keywords available"
    #print(keywords)

    try:
        url_paper = (soup.find(class_="obj_galley_link pdf") or soup.find(class_="obj_galley_link file"))
        publisher = url_paper.get_text(strip=True)
        #print(publisher)

        url_paper = url_paper.get('href')
        #print(url_paper)
    except AttributeError:
        publisher = "No publisher available"
        url_paper = "No URL available"

    try:
        date = soup.find(class_="item published")
        date = date.get_text(strip=True)
    except AttributeError:
        date = "No date available"
        #print(date)
    
    paper_number += 1

    df_papers = df_papers._append({'Event': event, 
                     'Authors': list_authors, 
                     'Title': title, 
                     'Abstract' : abstract, 
                     'Keywords' : keywords, 
                     'Publisher' : publisher, 
                     'URL_Paper' : url_paper,
                     'Date': date}, 
                     
                     ignore_index=True)

df_papers.to_csv('df_papers_caixa_1.csv', index=True)

https://sol.sbc.org.br/index.php/sbsc/article/view/24193
Paper number: 1
https://sol.sbc.org.br/index.php/ladc_estendido/article/view/39163
Paper number: 2
https://sol.sbc.org.br/index.php/ladc_estendido/article/view/39166
Paper number: 3
https://sol.sbc.org.br/index.php/ladc_estendido/article/view/39172
Paper number: 4
https://sol.sbc.org.br/index.php/ladc_estendido/article/view/39167
Paper number: 5
https://sol.sbc.org.br/index.php/ladc_estendido/article/view/39171
Paper number: 6
https://sol.sbc.org.br/index.php/sbrlars_estendido/article/view/39151
Paper number: 7
https://sol.sbc.org.br/index.php/sbrlars_estendido/article/view/39152
Paper number: 8
https://sol.sbc.org.br/index.php/sbrlars_estendido/article/view/39153
Paper number: 9
https://sol.sbc.org.br/index.php/sbrlars_estendido/article/view/39154
Paper number: 10
https://sol.sbc.org.br/index.php/sbrlars_estendido/article/view/39155
Paper number: 11
https://sol.sbc.org.br/index.php/sbrlars/article/view/39265
Paper number: 12
htt

In [2]:
import pandas as pd
import os
import glob

pasta = "C:\\Users\\Thiago Lobo\\Projeto-Mestrado\\Experimentos"
caminhos_arquivos = glob.glob(os.path.join(pasta, "*.csv"))

lista_df = []

for caminho_arquivo in caminhos_arquivos:
    df_individual = pd.read_csv(caminho_arquivo)
    lista_df.append(df_individual)

# 5. Concatene todos os DataFrames da lista em um único DataFrame
df_final = pd.concat(lista_df, ignore_index=True)

# 6. Opcional: Salve o resultado em um novo arquivo CSV
df_final.to_csv("arquivo_concatenado.csv", index=False)

# 7. Visualize o DataFrame final
print(df_final.head())
print(f"Total de linhas combinadas: {len(df_final)}")    

   Unnamed: 0                                              Event  \
0           0  \n\n\n\n\t\t\t\tInício\n\t\t\t\n/\n\n\n\n\t\t\...   
1           1  \n\n\n\n\t\t\t\tInício\n\t\t\t\n/\n\n\n\n\t\t\...   
2           2  \n\n\n\n\t\t\t\tInício\n\t\t\t\n/\n\n\n\n\t\t\...   
3           3  \n\n\n\n\t\t\t\tInício\n\t\t\t\n/\n\n\n\n\t\t\...   
4           4  \n\n\n\n\t\t\t\tInício\n\t\t\t\n/\n\n\n\n\t\t\...   

                                             Authors  \
0  Ana Paula ChavesNorthern Arizona UniversityDen...   
1                     Allan Edgard Silva FreitasIFBA   
2  Lenin ChacónTecnológico de Costa RicaKevin Mor...   
3  Pablo BlancoUniversidad de la RepúblicaGustavo...   
4  Carlos CardosoUNIFAVIP / UFPAhttps://orcid.org...   

                                               Title  \
0                         Apresentação e Organização   
1                    A DAG-Based Post-Quantum Ledger   
2  AI resources governance with OpenDID: Strategy...   
3  Digital Academic Certificat

In [ ]:
df_papers.head()

In [ ]:
pdf_link = "https://sol.sbc.org.br/index.php/erbd/article/download/24354/24177"

pdf_bytes = requests.get(pdf_link).content
with open("artigo_29986.pdf", "wb") as f:
    f.write(pdf_bytes)